# Lab 19-09 · Euler Paths, Pitches and Area

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import layout as ly

T = lambda name: ("t", name)
S = lambda *nets: ("s", list(nets))
P = lambda *nets: ("p", list(nets))

## Part 1 · NAND, NOR and complex gates

Each gate is given by its pull-down network; the pull-up is its dual. *Predict* the number of input orderings that are Euler paths in both networks, then run.

In [2]:
gates = {
    "NAND3": S(T("A"), T("B"), T("C")),
    "NOR3": P(T("A"), T("B"), T("C")),
    "OAI21": S(T("A"), P(T("B"), T("C"))),
    "AOI22": P(S(T("A"), T("B")), S(T("C"), T("D"))),
    "AOI211": P(S(T("A"), T("B")), T("C"), T("D")),
    "OAI22": S(P(T("A"), T("B")), P(T("C"), T("D"))),
}
for name, pdn in gates.items():
    orders = ly.common_euler_orders(pdn)
    print(f"{name:7s} {len(orders):2d} valid, e.g. {'-'.join(orders[0])}")

NAND3    2 valid, e.g. A-B-C
NOR3     2 valid, e.g. A-B-C
OAI21    4 valid, e.g. A-B-C
AOI22    8 valid, e.g. A-B-D-C
AOI211   4 valid, e.g. A-B-C-D
OAI22    8 valid, e.g. A-B-D-C


## Part 2 · Breaks

*Predict* how many diffusion breaks the AOI22 orderings A-B-C-D and A-C-B-D need in each network (stacks drawn as written, A and C nearest the output).

In [3]:
import itertools
aoi22 = gates["AOI22"]
e_n = ly.to_edges(aoi22, "out", "gnd", itertools.count(100))
e_p = ly.to_edges(ly.dual(aoi22), "out", "vdd", itertools.count(200))
for order in ("ABCD", "ACBD", "ABDC"):
    print(order, "pull-down breaks:", ly.diffusion_breaks(e_n, list(order)), " pull-up breaks:", ly.diffusion_breaks(e_p, list(order)))

ABCD pull-down breaks: 1  pull-up breaks: 0
ACBD pull-down breaks: 1  pull-up breaks: 1
ABDC pull-down breaks: 0  pull-up breaks: 0


## Part 3 · Larger gates

*Predict* the number of valid orderings of $\overline{AB + C(D + E)}$. Then predict whether $\overline{A(BC + DE)}$ has a common Euler path for *any* stack order, and the fewest breaks its best ordering needs.

In [4]:
big = P(S(T("A"), T("B")), S(T("C"), P(T("D"), T("E"))))
print("AB + C(D + E):", len(ly.common_euler_orders(big)), "valid orderings of 120")

hard = S(T("A"), P(S(T("B"), T("C")), S(T("D"), T("E"))))
variants = list(ly.stack_variants(hard))
print("A(BC + DE): pull-down stack orders", len(variants), ", pull-up stack orders", len(list(ly.stack_variants(ly.dual(hard)))))
count = 0
for pd in variants:
    e_n = ly.to_edges(pd, "out", "gnd", itertools.count(100))
    for pu in ly.stack_variants(ly.dual(hard)):
        e_p = ly.to_edges(pu, "out", "vdd", itertools.count(200))
        count += sum(1 for o in itertools.permutations("ABCDE")
                     if ly.is_euler_path(e_n, list(o)) and ly.is_euler_path(e_p, list(o)))
print("common Euler paths over all 16 stack-order combinations:", count)
breaks, order = ly.fewest_breaks(hard)
print("fewest total breaks:", breaks, "with ordering", "-".join(order))

AB + C(D + E): 6 valid orderings of 120
A(BC + DE): pull-down stack orders 8 , pull-up stack orders 2
common Euler paths over all 16 stack-order combinations: 0
fewest total breaks: 1 with ordering A-B-C-D-E


## Part 4 · Rules

*Predict* both pitches and the three-transistor strip lengths when contact-to-gate spacing grows from 2λ to 3λ.

In [5]:
for c2g in (2, 3):
    r = dict(ly.RULES, contact_to_poly=c2g)
    print(f"contact-to-gate {c2g} lambda: contacted pitch {ly.contacted_pitch(r)}, uncontacted {ly.uncontacted_pitch(r)}, "
          f"strip {ly.strip_length(3, False, r)} (end contacts only) or {ly.strip_length(3, True, r)} (all contacted) lambda")

contact-to-gate 2 lambda: contacted pitch 8, uncontacted 5, strip 22 (end contacts only) or 28 (all contacted) lambda
contact-to-gate 3 lambda: contacted pitch 10, uncontacted 5, strip 24 (end contacts only) or 34 (all contacted) lambda


## Part 5 · Area

*Predict* the area of a block with 500 kGates of logic (120 kGates/mm² routed) and 30 000 flip-flops of 20.02 µm² at 65% and 80% utilization.

In [6]:
logic = ly.gates_area_mm2(500, 120)
ff_cells = 30_000 * 20.02e-6
print(f"logic {logic:.2f} mm^2; flip-flop cells {ff_cells:.4f} mm^2")
utils = [0.5 + 0.025 * k for k in range(15)]
plt.plot([100 * u for u in utils], [ly.core_area(ff_cells, u) for u in utils])
for u in (0.65, 0.80):
    print(f"utilization {u:.0%}: flip-flop core area {ly.core_area(ff_cells, u):.3f} mm^2")
plt.xlabel("utilization (%)"); plt.ylabel("flip-flop core area (mm$^2$)"); plt.show()

logic 4.17 mm^2; flip-flop cells 0.6006 mm^2
utilization 65%: flip-flop core area 0.924 mm^2
utilization 80%: flip-flop core area 0.751 mm^2


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).